## Example 1: Zonal statistics for one polygon

In [ ]:
# rasterio, shapely and numpy are all rasterstats dependencies
import os
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import Polygon, mapping
from rasterstats import zonal_stats

# Example 1: Zonal statistics for one polygon
print("Example 1: Zonal statistics for one polygon")

try:
    # A 10 x 10 elevation raster whose value is row * 10 + column (0 .. 99), with 1 x 1 pixels
    # covering x 0..10 and y 0..10 (the top-left corner is at x=0, y=10)
    temp_dir = tempfile.mkdtemp()
    elevation = np.arange(100, dtype='float32').reshape(10, 10)
    transform = from_origin(0, 10, 1, 1)
    raster_path = os.path.join(temp_dir, 'elevation.tif')
    with rasterio.open(raster_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(elevation, 1)
    print(f"Wrote {raster_path}: 10x10, values {elevation.min():.0f}..{elevation.max():.0f}")


    # The square x 2..7, y 2..7 holds the pixel centres of rows 3..7 and columns 2..6: 25 pixels
    polygon = Polygon([(2, 2), (2, 7), (7, 7), (7, 2)])
    stats = zonal_stats(polygon, raster_path, stats=['count', 'min', 'max', 'mean', 'sum'])
    print(f"Polygon area: {polygon.area:.1f}, zonal statistics: {stats}")
    assert len(stats) == 1, stats
    assert stats[0] == {'count': 25, 'min': 32.0, 'max': 76.0, 'mean': 54.0, 'sum': 1350.0}, stats

    # A GeoJSON-like mapping works the same as a shapely geometry
    assert zonal_stats(mapping(polygon), raster_path, stats='count')[0]['count'] == 25

    # An in-memory array plus its affine transform gives the same result as the file
    from_array = zonal_stats(polygon, elevation, affine=transform, stats='count mean', nodata=-9999)
    print(f"From the in-memory array: {from_array}")
    assert from_array == [{'count': 25, 'mean': 54.0}], from_array

    print("Zonal statistics for one polygon completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rasterstats example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Multiple geometries and statistic choices

In [ ]:
# rasterio, shapely and numpy are all rasterstats dependencies
import os
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
import math
from shapely.geometry import Polygon
from rasterstats import zonal_stats

# Example 2: Multiple geometries and statistic choices
print("Example 2: Multiple geometries and statistic choices")

try:
    # A 10 x 10 elevation raster whose value is row * 10 + column (0 .. 99), with 1 x 1 pixels
    # covering x 0..10 and y 0..10 (the top-left corner is at x=0, y=10)
    temp_dir = tempfile.mkdtemp()
    elevation = np.arange(100, dtype='float32').reshape(10, 10)
    transform = from_origin(0, 10, 1, 1)
    raster_path = os.path.join(temp_dir, 'elevation.tif')
    with rasterio.open(raster_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(elevation, 1)
    print(f"Wrote {raster_path}: 10x10, values {elevation.min():.0f}..{elevation.max():.0f}")


    polygons = [
        Polygon([(0, 0), (3, 0), (3, 3), (0, 3)]),  # bottom-left 3 x 3 square
        Polygon([(5, 5), (8, 5), (8, 8), (5, 8)]),  # 3 x 3 square in the upper middle
        Polygon([(1, 6), (4, 7), (3, 9), (0, 8)]),  # irregular quadrilateral
    ]
    stats = zonal_stats(polygons, raster_path, stats='count sum median range unique majority minority std')
    for i, s in enumerate(stats, 1):
        print(f"  Polygon {i}: {s}")
    assert [s['count'] for s in stats] == [9, 9, 7], stats
    assert [s['sum'] for s in stats] == [729.0, 324.0, 161.0], stats
    assert [s['median'] for s in stats] == [81.0, 36.0, 22.0], stats
    assert [s['range'] for s in stats] == [22.0, 22.0, 20.0], stats
    # every pixel value is distinct, so each value is unique and the smallest wins the majority/minority tie
    assert [s['unique'] for s in stats] == [9, 9, 7], stats
    assert [(s['majority'], s['minority']) for s in stats] == [(70.0, 70.0), (25.0, 25.0), (12.0, 12.0)], stats
    assert math.isclose(stats[0]['std'], np.std([70, 71, 72, 80, 81, 82, 90, 91, 92]), rel_tol=1e-6), stats[0]

    # stats='*' asks for every built-in statistic
    everything = zonal_stats(polygons[0], raster_path, stats='*')[0]
    print(f"  All statistics: {sorted(everything)}")
    assert {'count', 'min', 'max', 'mean', 'sum', 'std', 'median', 'majority', 'minority', 'unique', 'range'} <= set(everything)

    # add_stats takes custom functions of the masked pixel array; prefix renames the output keys
    custom = zonal_stats(polygons[0], raster_path, stats='count',
                         add_stats={'share_even': lambda x: float((x % 2 == 0).sum()) / x.count()}, prefix='elev_')
    print(f"  Custom statistic: {custom}")
    assert custom[0]['elev_count'] == 9 and math.isclose(custom[0]['elev_share_even'], 6 / 9), custom

    print("Multiple geometries and statistic choices completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rasterstats example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Point queries

In [ ]:
# rasterio, shapely and numpy are all rasterstats dependencies
import os
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import Point
from rasterstats import point_query, gen_point_query

# Example 3: Point queries
print("Example 3: Point queries")

try:
    # A 10 x 10 elevation raster whose value is row * 10 + column (0 .. 99), with 1 x 1 pixels
    # covering x 0..10 and y 0..10 (the top-left corner is at x=0, y=10)
    temp_dir = tempfile.mkdtemp()
    elevation = np.arange(100, dtype='float32').reshape(10, 10)
    transform = from_origin(0, 10, 1, 1)
    raster_path = os.path.join(temp_dir, 'elevation.tif')
    with rasterio.open(raster_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(elevation, 1)
    print(f"Wrote {raster_path}: 10x10, values {elevation.min():.0f}..{elevation.max():.0f}")


    # Points at pixel centres read that pixel's value; a point outside the raster gives None
    points = [Point(2.5, 7.5), Point(0.5, 0.5), Point(9.5, 9.5), Point(20, 20)]
    values = point_query(points, raster_path)
    print(f"  Values at pixel centres: {values}")
    assert values == [22.0, 90.0, 9.0, None], values

    # Between pixel centres the default is bilinear interpolation; 'nearest' takes one pixel
    corner = Point(5, 5)  # the corner shared by pixels 44, 45, 54 and 55
    bilinear = point_query(corner, raster_path)[0]
    nearest = point_query(corner, raster_path, interpolate='nearest')[0]
    print(f"  At (5, 5): bilinear={bilinear}, nearest={nearest}")
    assert bilinear == 49.5 and nearest in (44.0, 45.0, 54.0, 55.0), (bilinear, nearest)

    # gen_point_query is the lazy (generator) form
    assert list(gen_point_query(points[:2], raster_path)) == [22.0, 90.0]

    # With geojson_out=True the value is added to each feature's properties
    feature = {'type': 'Feature', 'properties': {'site': 'A'}, 'geometry': {'type': 'Point', 'coordinates': (2.5, 7.5)}}
    out = point_query(feature, raster_path, geojson_out=True)
    print(f"  GeoJSON output properties: {out[0]['properties']}")
    assert out[0]['properties'] == {'site': 'A', 'value': 22.0}, out

    print("Point queries completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rasterstats example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Categorical rasters and multi-part geometries

In [ ]:
# rasterio, shapely and numpy are all rasterstats dependencies
import os
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import Polygon, MultiPolygon
from rasterstats import zonal_stats

# Example 4: Categorical rasters and multi-part geometries
print("Example 4: Categorical rasters and multi-part geometries")

try:
    # A 10 x 10 elevation raster whose value is row * 10 + column (0 .. 99), with 1 x 1 pixels
    # covering x 0..10 and y 0..10 (the top-left corner is at x=0, y=10)
    temp_dir = tempfile.mkdtemp()
    elevation = np.arange(100, dtype='float32').reshape(10, 10)
    transform = from_origin(0, 10, 1, 1)
    raster_path = os.path.join(temp_dir, 'elevation.tif')
    with rasterio.open(raster_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(elevation, 1)
    print(f"Wrote {raster_path}: 10x10, values {elevation.min():.0f}..{elevation.max():.0f}")


    # A land-cover raster: class codes 1..4, and 0 as nodata
    landcover = np.array([[1, 1, 1, 1, 1, 2, 2, 2, 2, 2]] * 5 + [[3, 3, 3, 4, 4, 4, 4, 4, 1, 1]] * 5, dtype='uint8')
    landcover_path = os.path.join(temp_dir, 'landcover.tif')
    with rasterio.open(landcover_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='uint8',
                       crs='EPSG:3857', transform=transform, nodata=0) as dst:
        dst.write(landcover, 1)
    class_names = {1: 'Forest', 2: 'Water', 3: 'Urban', 4: 'Agriculture'}

    # categorical=True returns a pixel count per class value
    whole_extent = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])
    counts = zonal_stats(whole_extent, landcover_path, categorical=True)[0]
    print(f"  Pixel counts per class: {counts}")
    assert counts == {1: 35, 2: 25, 3: 15, 4: 25}, counts

    # category_map replaces the class values with names
    named = zonal_stats(whole_extent, landcover_path, categorical=True, category_map=class_names)[0]
    print(f"  Named: {named}")
    assert named == {'Forest': 35, 'Water': 25, 'Urban': 15, 'Agriculture': 25}, named
    shares = {name: count / sum(named.values()) * 100 for name, count in named.items()}
    print(f"  Shares: {', '.join(f'{k} {v:.0f}%' for k, v in shares.items())}")

    # The top-left quarter is all forest; categorical counts can be combined with other statistics
    top_left = Polygon([(0, 5), (5, 5), (5, 10), (0, 10)])
    quarter = zonal_stats(top_left, landcover_path, categorical=True, category_map=class_names,
                          stats=['majority', 'unique'])[0]
    print(f"  Top-left quarter: {quarter}")
    assert quarter == {'Forest': 25, 'majority': 1.0, 'unique': 1}, quarter

    # A MultiPolygon is treated as one zone: its parts' pixels are pooled
    parts = MultiPolygon([Polygon([(1, 1), (3, 1), (3, 3), (1, 3)]), Polygon([(5, 5), (7, 5), (7, 7), (5, 7)])])
    pooled = zonal_stats(parts, raster_path, stats='count min max mean')[0]
    print(f"  MultiPolygon with {len(parts.geoms)} parts: {pooled}")
    assert pooled == {'count': 8, 'min': 35.0, 'max': 82.0, 'mean': 58.5}, pooled

    print("Categorical rasters and multi-part geometries completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rasterstats example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: GeoJSON features, nodata and rasterisation options

In [ ]:
# rasterio, shapely and numpy are all rasterstats dependencies
import os
import tempfile
import numpy as np
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import Polygon, mapping
from rasterstats import zonal_stats, gen_zonal_stats

# Example 5: GeoJSON features, nodata and rasterisation options
print("Example 5: GeoJSON features, nodata and rasterisation options")

try:
    # A 10 x 10 elevation raster whose value is row * 10 + column (0 .. 99), with 1 x 1 pixels
    # covering x 0..10 and y 0..10 (the top-left corner is at x=0, y=10)
    temp_dir = tempfile.mkdtemp()
    elevation = np.arange(100, dtype='float32').reshape(10, 10)
    transform = from_origin(0, 10, 1, 1)
    raster_path = os.path.join(temp_dir, 'elevation.tif')
    with rasterio.open(raster_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(elevation, 1)
    print(f"Wrote {raster_path}: 10x10, values {elevation.min():.0f}..{elevation.max():.0f}")


    zones = [
        Polygon([(0, 0), (3, 0), (3, 3), (0, 3)]),
        Polygon([(5, 5), (8, 5), (8, 8), (5, 8)]),
        Polygon([(1, 6), (4, 7), (3, 9), (0, 8)]),
    ]
    collection = {
        'type': 'FeatureCollection',
        'features': [{'type': 'Feature', 'properties': {'name': f'zone{i}'}, 'geometry': mapping(z)}
                     for i, z in enumerate(zones)],
    }

    # geojson_out=True returns the input features with the statistics added to their properties
    features = zonal_stats(collection, raster_path, stats='mean', geojson_out=True)
    print(f"  Feature properties: {[f['properties'] for f in features]}")
    assert [f['type'] for f in features] == ['Feature'] * 3
    assert [f['properties'] for f in features] == [
        {'name': 'zone0', 'mean': 81.0}, {'name': 'zone1', 'mean': 36.0}, {'name': 'zone2', 'mean': 23.0}], features

    # gen_zonal_stats yields one result at a time, for large feature collections
    generator = gen_zonal_stats(zones, raster_path, stats='count')
    assert not isinstance(generator, list)
    assert list(generator) == [{'count': 9}, {'count': 9}, {'count': 7}]

    # Pixels equal to the raster's nodata value are left out of the statistics and counted separately
    with_gap = elevation.copy()
    with_gap[3, 2] = -9999  # the zone's minimum pixel (value 32)
    gap_path = os.path.join(temp_dir, 'with_gap.tif')
    with rasterio.open(gap_path, 'w', driver='GTiff', height=10, width=10, count=1, dtype='float32',
                       crs='EPSG:3857', transform=transform, nodata=-9999) as dst:
        dst.write(with_gap, 1)
    zone = Polygon([(2, 2), (2, 7), (7, 7), (7, 2)])
    gap_stats = zonal_stats(zone, gap_path, stats='count nodata min')[0]
    print(f"  With one nodata pixel: {gap_stats}")
    assert gap_stats == {'count': 24, 'nodata': 1.0, 'min': 33.0}, gap_stats

    # By default a pixel belongs to a polygon when its centre does; all_touched=True takes every touched pixel
    small = Polygon([(2.6, 2.6), (6.4, 2.6), (6.4, 6.4), (2.6, 6.4)])
    centres = zonal_stats(small, raster_path, stats='count min max')[0]
    touched = zonal_stats(small, raster_path, stats='count min max', all_touched=True)[0]
    print(f"  Centre rule: {centres}; all_touched: {touched}")
    assert centres == {'count': 9, 'min': 43.0, 'max': 65.0}, centres
    assert touched == {'count': 25, 'min': 32.0, 'max': 76.0}, touched

    # raster_out=True also returns the clipped pixel window and its transform
    window = zonal_stats(zone, raster_path, stats='count', raster_out=True)[0]
    print(f"  Clipped window: {window['mini_raster_array'].shape}, origin ({window['mini_raster_affine'].c}, {window['mini_raster_affine'].f})")
    assert window['mini_raster_array'].shape == (5, 5)
    assert (window['mini_raster_affine'].c, window['mini_raster_affine'].f) == (2.0, 7.0)
    assert (window['mini_raster_array'].min(), window['mini_raster_array'].max()) == (32.0, 76.0)

    print("GeoJSON features, nodata and rasterisation options completed")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"rasterstats example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e